# RAMP-RAG: Clean-Stream Fusion & Empirical Robustness Evaluation
**Paper:** *Multi-Stage Multimodal RAG with Text-Proxy Hybrid Retrieval*
**Evaluation Protocol:** Strict 50/50 Dev/Test partition by Image ID, zero target leakage, observable reliability (zero oracle), multi-seed robustness (42, 43, 44), 2,000 bootstrap resamples, Clopper-Pearson CIs, paired permutation tests with Holm-Bonferroni correction, and standardized 3-run latency profiling on warm cache.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

REPO_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
DATA_DIR = REPO_ROOT / "ai_engine" / "rag" / "data"
RESULTS_JSON = DATA_DIR / "clean_stream_experiments_results.json"
PLOT_PNG = DATA_DIR / "mrr_hit10_vs_noise.png"

with open(RESULTS_JSON, encoding="utf-8") as f:
    results = json.load(f)
print("Benchmark results successfully loaded!")
print(f"Dev-Tuned Gamma*: {results.get('gamma_star_dev')}")

## Task 1: Dev Tuning for Clean-Stream Fusion (Dev Split Only)
Grid search over $\gamma \in [0.0, 0.8]$ under 20% proxy noise on the 96-item Dev corpus and 30 Dev probes.

In [2]:
dev_gamma = results["gamma_sweep_dev"]
rows = []
for g, metrics in sorted(dev_gamma.items(), key=lambda x: float(x[0])):
    rows.append({"Gamma": float(g), "Dev Hit@10": metrics["hit10"], "Dev MRR": metrics["mrr"]})
df_gamma = pd.DataFrame(rows)
display(df_gamma)
print(f"Selected Gamma*: {results['gamma_star_dev']}")

## Task 2: Multi-Seed Noise Robustness Sweep (Held-Out Test Split)
Evaluation across character corruption rates (0%, 10%, 20%, 40%) across seeds 42, 43, 44.

In [3]:
t2 = results["task2_noise_sweep"]
t2_rows = []
for noise_key in ["noise_0pct", "noise_10pct", "noise_20pct", "noise_40pct"]:
    pct = noise_key.replace("noise_", "").replace("pct", "%")
    for method, d in t2[noise_key].items():
        t2_rows.append({
            "Noise": pct,
            "Method": method,
            "Hit@1": d["hit1"],
            "Hit@5": d["hit5"],
            "Hit@10": d["hit10"],
            "Hit@10_95CI": str(d["hit10_ci95"]),
            "MRR": d["mrr"],
            "MRR_95CI": str(d["mrr_ci95"]),
            "NDCG@10": d["ndcg10"],
            "Holm_p_vs_Hybrid": d.get("p_value_mrr_vs_hybrid", 1.0),
        })
df_t2 = pd.DataFrame(t2_rows)
display(df_t2)
if PLOT_PNG.exists():
    display(Image(filename=str(PLOT_PNG)))

## Task 3: Conclusive 5-Row Component Ablation
Evaluated across COCO Test (N=250), Clean Scene-Text Test (N=55), and 20% Noise Scene-Text Test (N=55 x 3 seeds).

In [4]:
t3 = results["task3_ablation"]
t3_rows = []
for row_name in ["Baseline (Fixed alpha=0.7)", "+ Router (Learned alpha_q)", "+ Cascade (tau=0.065)", "+ Reliability (Observable r_m)", "Full RAMP-RAG (+ Clean Stream)"]:
    coco = t3["coco_test"].get(row_name, {})
    clean = t3["ocr_clean_test"].get(row_name, {})
    n20 = t3["ocr_noise20_test"].get(row_name, {})
    t3_rows.append({
        "Component": row_name,
        "COCO Hit@10": coco.get("hit10"),
        "COCO MRR": coco.get("mrr"),
        "Clean Hit@10": clean.get("hit10"),
        "Clean MRR": clean.get("mrr"),
        "Noise20 Hit@10": n20.get("hit10"),
        "Noise20 Hit@10_CI": str(n20.get("hit10_ci95")),
        "Noise20 MRR": n20.get("mrr"),
        "Noise20 MRR_CI": str(n20.get("mrr_ci95")),
        "p_val_MRR": n20.get("p_val_mrr"),
    })
df_t3 = pd.DataFrame(t3_rows)
display(df_t3)

## Task 4: COCO with Realistic BLIP Proxies & Stronger Baselines
Comparing BLIP caption proxies, BGE-base-en-v1.5, and CLIP zero-shot visual retrieval on 1,000 images.

In [5]:
t4 = results["task4_coco_benchmarks"]
t4_rows = []
for m, d in t4.items():
    t4_rows.append({
        "Method": m,
        "Hit@1": d["hit1"],
        "Hit@5": d["hit5"],
        "Hit@10": d["hit10"],
        "Hit@10_95CI": str(d["hit10_ci95"]),
        "MRR": d["mrr"],
        "MRR_95CI": str(d["mrr_ci95"]),
        "NDCG@10": d["ndcg10"],
    })
df_t4 = pd.DataFrame(t4_rows)
display(df_t4)

## Task 5: Standardized Latency Profile
End-to-end scope (query encoding + retrieval + rerank), warm cache, mean and p95 over 3 runs.

In [6]:
t5 = results["task5_latencies"]
t5_rows = [{"System": s, "Mean_ms": d["mean_ms"], "P95_ms": d["p95_ms"]} for s, d in t5.items()]
df_t5 = pd.DataFrame(t5_rows)
display(df_t5)

## Core Empirical Decision
**Question:** Does RAMP-RAG + clean stream beat unweighted hybrid at 20% and 40% noise with non-overlapping CIs and a Holm-corrected p < 0.05?
**Plain Answer: NO.**
- At 20% noise: Hybrid MRR = 0.8254 [0.7769, 0.8722] vs RAMP-RAG Clean = 0.8624 [0.8210, 0.9003]. The CIs overlap, and Holm p = 0.3252 (> 0.05).
- At 40% noise: Hybrid MRR = 0.6677 [0.6048, 0.7271] vs RAMP-RAG Clean = 0.7802 [0.7263, 0.8306]. Holm p = 0.0084, but CIs touch/overlap at 0.7263-0.7271.
- Critical Finding: BM25 alone achieves MRR = 0.9473 at 20% noise and 0.8993 at 40% noise, outperforming ALL hybrid and dense methods.